# Introduction to AI and LLMs for Molecular Dynamics

Molecular dynamics (MD) uses physical models to simulate how atoms and molecules move. Artificial intelligence (AI) can help MD workflows by learning patterns from data, accelerating expensive calculations, analyzing trajectories, and helping researchers interact with complex simulation results.

This notebook introduces the basics of AI and large language models (LLMs) in the context of molecular dynamics.

By the end, you should be able to describe:

1. What kinds of MD tasks AI can support.
2. The difference between physics-based models, machine learning models, and LLMs.
3. How a simple model can learn an approximate force law from data.
4. Where LLMs are useful and where they should be used carefully.

## 1. Why AI Matters for Molecular Dynamics

MD simulations can generate large amounts of data, and accurate simulations can be computationally expensive. AI methods are useful when they help with one of these bottlenecks.

Common AI-assisted MD tasks include:

- Learning fast surrogate models for expensive potential energy calculations.
- Predicting forces, energies, or material properties from atomic structures.
- Detecting states, transitions, clusters, or rare events in trajectories.
- Selecting new simulations to run through active learning.
- Summarizing simulation results and helping generate analysis code.

AI does not replace physical judgment. A useful AI model still needs good training data, careful validation, and awareness of the chemistry or materials system being studied.

## 2. Three Model Types

It helps to distinguish three broad kinds of models.

| Model type | What it uses | Example role in MD |
|---|---|---|
| Physics-based model | Equations and parameters chosen by humans | Classical force field or quantum calculation |
| Machine learning model | Patterns learned from numerical data | Neural network potential or trajectory classifier |
| Large language model | Patterns learned from text and code | Explaining concepts, writing scripts, summarizing papers, planning analyses |

A modern AI-MD workflow may use all three: physics to define the problem, machine learning to accelerate or analyze it, and LLMs to help researchers work with code, documentation, and results.

## 3. Toy Example: Learning a Force Law

In real AI-MD, a model might learn energies and forces from quantum mechanical calculations. Here we will use a much simpler example.

We will generate training data from the Lennard-Jones force and fit a small linear model using hand-designed features. This is not a production-quality model, but it illustrates the core idea: a model can learn a relationship between molecular geometry and force.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
def lj_force_magnitude(r, epsilon=1.0, sigma=1.0):
    """Lennard-Jones force magnitude in reduced units."""
    sr6 = (sigma / r) ** 6
    return 24 * epsilon * (2 * sr6 ** 2 - sr6) / r


rng = np.random.default_rng(7)

# Training separations. Avoid very small r, where the force changes extremely fast.
r_train = np.linspace(0.95, 3.0, 120)
force_train = lj_force_magnitude(r_train)

# Add a little noise to mimic imperfect training data.
force_train_noisy = force_train + rng.normal(scale=0.15, size=r_train.shape)

A machine learning model needs a representation of the input. For this toy model, we will use powers of inverse distance as features:

$$x(r) = \left[1, r^{-1}, r^{-2}, r^{-6}, r^{-7}, r^{-12}, r^{-13}\right]$$

These features are chosen because the Lennard-Jones force has strong inverse-power behavior. In real AI-MD models, representations are usually designed to respect physical symmetries such as translation, rotation, and permutation of identical atoms.

In [ ]:
def make_features(r):
    r = np.asarray(r)
    return np.column_stack([
        np.ones_like(r),
        r ** -1,
        r ** -2,
        r ** -6,
        r ** -7,
        r ** -12,
        r ** -13,
    ])


X_train = make_features(r_train)

# Least-squares fit: find weights that map features to force.
weights, *_ = np.linalg.lstsq(X_train, force_train_noisy, rcond=None)

def learned_force(r):
    return make_features(r) @ weights

In [ ]:
r_test = np.linspace(0.9, 3.2, 300)
true_force = lj_force_magnitude(r_test)
predicted_force = learned_force(r_test)

plt.scatter(r_train, force_train_noisy, s=18, alpha=0.5, label="training data")
plt.plot(r_test, true_force, label="true force")
plt.plot(r_test, predicted_force, "--", label="learned force")
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Separation, r")
plt.ylabel("Force magnitude")
plt.title("Learning an Approximate Pair Force")
plt.ylim(-3, 8)
plt.legend()
plt.show()

In [ ]:
rmse = np.sqrt(np.mean((predicted_force - true_force) ** 2))
print(f"Test RMSE over the plotted range: {rmse:.3f}")

## 4. What This Toy Example Teaches

The model above is deliberately simple, but it reveals several important AI-MD lessons:

- Training data defines what the model can learn.
- Feature choice matters.
- A model can interpolate well but extrapolate poorly outside its training range.
- No model should be trusted just because it produces a smooth curve.
- Validation should measure physically meaningful quantities, not only numerical error.

For real molecular dynamics, AI potentials must be checked for energy conservation, stable trajectories, correct structures, reasonable thermodynamics, and transferability to new configurations.

## 5. Where LLMs Fit

Large language models are not the same as force fields or neural network potentials. LLMs are usually most useful around the simulation workflow rather than inside the inner MD time-integration loop.

Useful LLM tasks include:

- Explaining MD concepts and simulation parameters.
- Drafting analysis scripts for trajectories.
- Helping convert file formats or organize workflows.
- Summarizing papers, documentation, and simulation logs.
- Generating hypotheses or checklists for validation.
- Helping build teaching materials and notebooks.

LLMs can make mistakes, especially with exact syntax, package APIs, citations, and domain-specific claims. Treat LLM output as a draft that needs testing and expert review.

## 6. Example Prompts for AI-MD Workflows

Here are examples of useful LLM prompts for molecular dynamics work:

- Explain the difference between NVE, NVT, and NPT ensembles for an undergraduate chemistry student.
- Write Python code to plot temperature and total energy from a simulation log file.
- Given this trajectory analysis result, suggest checks for whether the simulation is equilibrated.
- Summarize the limitations of using a machine-learned potential outside its training domain.
- Turn this notebook cell into a short guided-inquiry question for students.

Good prompts include context, the intended audience, the desired output format, and constraints such as packages to use or assumptions to avoid.

## 7. Practical Checklist

When using AI or LLMs with molecular dynamics, ask:

- What data was used to train the model?
- Is the model being used inside or outside its training domain?
- Which physical symmetries are built into the model?
- What validation tests were performed?
- Are uncertainty estimates or failure checks available?
- Can the result be reproduced without the LLM?
- Has any generated code been run and inspected?

AI can be a powerful partner for MD, but the final responsibility remains scientific: understand the assumptions, validate the outputs, and keep the physics in view.